# 07 · Topic deep dive — BERTopic (historical / comparison-only as of 2026-08-29)

Deep dive into BERTopic (SPECTER2 → UMAP → HDBSCAN → c-TF-IDF), fitted offline by
[`src/topics_bertopic.py`](../src/topics_bertopic.py) at `min_cluster_size=20, seed=42`
on the M2-enriched, backfilled corpus (2,676 grants + 65 orphan pseudo-docs), refit
2026-08-20. **32 topics + an honest "Unassigned" (noise) cluster**, grouped under 8
parent themes (BERTopic's own, since retired — see below). LDA is retired too (kept
only in `src/topics_lda.py` for continuity).

**As of 2026-08-29, BERTopic is no longer the pipeline's canonical topic source** — a
curated, deterministic keyword classifier (`src/classify_by_keywords.py`, 31 leaves /
7 parents) replaced it; see CLAUDE.md's "Topic modeling — state of play". BERTopic's
own assignment (loaded below, unchanged) is kept as a comparison column
(`bertopicDom`/`bertopicNoise` downstream), not deleted, and this notebook's own
analysis of it is still historically accurate — it documents what BERTopic actually
produced, not a claim about what's canonical today.

Everything here **loads** the committed artifacts — no model is refit.
See [`TOPIC_WORK_EXECUTION_REPORT.md`](../docs/TOPIC_WORK_EXECUTION_REPORT.md).


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.cluster.hierarchy import linkage, dendrogram
from bertopic import BERTopic

sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 110

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'data' / 'processed').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PROCESSED = REPO_ROOT / 'data' / 'processed'
OUTPUTS   = REPO_ROOT / 'outputs'; OUTPUTS.mkdir(exist_ok=True)
DOCS      = REPO_ROOT / 'docs';    DOCS.mkdir(exist_ok=True)
FIG_DIR   = REPO_ROOT / 'notebooks' / 'figures'; FIG_DIR.mkdir(parents=True, exist_ok=True)
assert PROCESSED.exists()

## 0 · Load the BERTopic model + assignments + labels (historical/comparison — see the note above)


In [ ]:
faculty = pd.read_parquet(PROCESSED / 'faculty.parquet')
grants  = pd.read_parquet(PROCESSED / 'grants.parquet')
fg      = pd.read_parquet(PROCESSED / 'faculty_grants.parquet')
ta      = pd.read_parquet(PROCESSED / 'topic_assignments.parquet')
for d, cols in [(grants, ['grant_id']), (fg, ['grant_id', 'faculty_id']),
                (faculty, ['faculty_id']), (ta, ['doc_id'])]:
    for c in cols:
        d[c] = d[c].astype(str)

topic_model = BERTopic.load(str(PROCESSED / 'bertopic_model'))
labels  = json.load(open(OUTPUTS / 'topic_labels.json', encoding='utf-8'))
TL, PARENTS = labels['topics'], labels['parents']
id2label  = {int(k): v['label'] for k, v in TL.items()}
id2parent = {int(k): (PARENTS[v['parent']]['label'] if v['parent'] else None)
             for k, v in TL.items()}

# grant-level topic (exclude the orphan-* pseudo-docs), + lead-PI college
gt = ta[~ta['is_extra']][['doc_id', 'topic_id']].rename(columns={'doc_id': 'grant_id'})
g = grants.merge(gt, on='grant_id', how='left')
g['topic_id']    = g['topic_id'].fillna(-1).astype(int)
g['topic_label'] = g['topic_id'].map(id2label).fillna('Unassigned')
g['parent']      = g['topic_id'].map(id2parent).fillna('Unassigned')
lead = fg[fg['is_pi']].merge(faculty[['faculty_id', 'superior_academic_unit']],
                             on='faculty_id', how='left')
g['college'] = g['grant_id'].map(lead.groupby('grant_id')['superior_academic_unit'].first()).astype('object')

diag = json.load(open(OUTPUTS / 'bertopic_diagnostics.json'))
print(f"BERTopic: {diag['n_topics']} topics | {diag['pct_noise']}% noise "
      f"({diag['n_noise']}/{diag['n_docs']}) | intra-cosine {diag['mean_intra_cluster_cosine']}")
print(f"grants: {len(g)} | assigned to a topic: {(g.topic_id>=0).sum()} | "
      f"unassigned: {(g.topic_id==-1).sum()}")
display(topic_model.get_topic_info()[['Topic', 'Count', 'Name']].head(26))

## 1 · Topic prevalence

In [ ]:
sizes = g[g.topic_id >= 0].groupby('topic_id').size().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(range(len(sizes)), sizes.values, color='steelblue')
ax.set_yticks(range(len(sizes))); ax.set_yticklabels([id2label[t] for t in sizes.index], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('grants')
ax.set_title(f'BERTopic topic prevalence  (n={(g.topic_id>=0).sum()} assigned, '
             f'{(g.topic_id==-1).sum()} unassigned)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'w7_topic_prevalence.png', dpi=120, bbox_inches='tight'); plt.show()

## 2 · What does each college work on? (topics × college, row-normalised)

In [ ]:
sub = g[(g.topic_id >= 0) & g.college.notna()].copy()
top_colleges = sub.college.value_counts().head(8).index.tolist()
sub['col2'] = sub.college.where(sub.college.isin(top_colleges), 'Other')
ct  = pd.crosstab(sub.topic_label, sub.col2)
ctn = ct.div(ct.sum(axis=1), axis=0) * 100    # each topic sums to 100% across colleges
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(ctn, annot=True, fmt='.0f', cmap='YlOrRd', cbar_kws={'label': '% of topic'}, ax=ax)
ax.set_title('Topics × college (row-normalised, %)'); ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.savefig(FIG_DIR / 'w7_topic_by_college_rownorm.png', dpi=120, bbox_inches='tight'); plt.show()

## 3 · College profiles — top topics, agencies, PIs

In [ ]:
fg_full = fg.merge(g[['grant_id', 'topic_id', 'topic_label', 'college', 'totaldollars']],
                   on='grant_id', how='inner')
rows = []
for col in top_colleges:
    cg = g[(g.college == col) & (g.topic_id >= 0)]
    tt = cg.topic_label.value_counts().head(3)
    ta_ = cg.agencyname.value_counts().head(3)
    pis = (fg_full[(fg_full.college == col) & fg_full.is_pi]
           .groupby('faculty_name')['totaldollars'].sum().sort_values(ascending=False).head(5))
    rows.append({'college': col, 'n_grants': len(cg),
                 'total_$M': round(cg.totaldollars.sum() / 1e6, 1),
                 'top_topics': ' | '.join(f'{k} ({v})' for k, v in tt.items()),
                 'top_agencies': ' | '.join(f'{k} ({v})' for k, v in ta_.items()),
                 'top_PIs': ' | '.join(f'{k} (${v/1e6:.1f}M)' for k, v in pis.items())})
prof = pd.DataFrame(rows).sort_values('total_$M', ascending=False).reset_index(drop=True)
prof.to_csv(OUTPUTS / 'college_profiles.csv', index=False)
display(prof[['college', 'n_grants', 'total_$M', 'top_topics', 'top_agencies']])

## 4 · Topic hierarchy — 8 parent themes

In [ ]:
prows = []
for pid, p in PARENTS.items():
    tids = p['topic_ids']
    prows.append({'parent': p['label'], 'n_topics': len(tids),
                  'n_grants': int(g[g.topic_id.isin(tids)].shape[0]),
                  'topics': ', '.join(id2label[t] for t in tids)})
parent_df = pd.DataFrame(prows).sort_values('n_grants', ascending=False).reset_index(drop=True)
display(parent_df)
ps = parent_df.sort_values('n_grants')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ps.parent, ps.n_grants, color='teal')
ax.set_xlabel('grants'); ax.set_title('Grants per parent theme (8 super-groups)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'w7_parent_sizes.png', dpi=120, bbox_inches='tight'); plt.show()

## 5 · Topic similarity dendrogram (SPECTER2 centroids)

In [ ]:
emb = np.load(PROCESSED / 'specter2_embeddings.npy')
ids = (PROCESSED / 'specter2_ids.txt').read_text().splitlines()
id2row = {s: i for i, s in enumerate(ids)}
norm = emb / (np.linalg.norm(emb, axis=1, keepdims=True) + 1e-9)
tids = sorted(t for t in g.topic_id.unique() if t >= 0)
cent = np.vstack([norm[[id2row[d] for d in g[g.topic_id == t].grant_id if d in id2row]].mean(0)
                  for t in tids])
Z = linkage(cent, method='average', metric='cosine')
fig, ax = plt.subplots(figsize=(10, 8))
dendrogram(Z, labels=[id2label[t] for t in tids], orientation='right', leaf_font_size=8, ax=ax)
ax.set_title('BERTopic topic similarity (SPECTER2 centroids, cosine · average linkage)')
plt.tight_layout(); plt.savefig(FIG_DIR / 'w7_topic_dendrogram.png', dpi=120, bbox_inches='tight'); plt.show()

## 6 · UMAP projection (canonical SPECTER2 coords)

Uses the committed 2-D UMAP (`specter2_umap_2d.npy`) — the *same* coordinates the
`grant_atlas` app renders, so this lines up with the interactive viz.

In [ ]:
umap2d = np.load(PROCESSED / 'specter2_umap_2d.npy')
udf = pd.DataFrame({'doc_id': ids, 'x': umap2d[:, 0], 'y': umap2d[:, 1]})
udf = udf[~udf.doc_id.str.startswith('orphan-')]      # grants only
udf = udf.merge(g[['grant_id', 'topic_id', 'topic_label', 'parent', 'college',
                   'agencyname', 'startdateyear', 'grantname']],
                left_on='doc_id', right_on='grant_id', how='left')

def _bucket(a):
    a = str(a).lower()
    for key, lab in [('national science', 'NSF'), ('subaward', 'NIH-SUB'),
                     ('national institutes', 'NIH'), ('nav', 'Navy'), ('nasa', 'NASA'),
                     ('army', 'Army'), ('energy', 'DOE'), ('air force', 'AFRO')]:
        if key in a:
            return lab
    return 'Other'
udf['agency_grp'] = udf.agencyname.map(_bucket)
udf['college_grp'] = udf.college.where(udf.college.isin(top_colleges), 'Other')

def _scat(ax, col, title, cont=False):
    if cont:
        s = ax.scatter(udf.x, udf.y, c=udf[col], s=6, cmap='viridis', alpha=0.6)
        plt.colorbar(s, ax=ax, shrink=0.7)
    else:
        cats = udf[col].fillna('Unassigned').astype(str)
        order = cats.value_counts().index.tolist()
        pal = sns.color_palette('tab20', len(order))
        for c, color in zip(order, pal):
            m = cats == c
            ax.scatter(udf.loc[m, 'x'], udf.loc[m, 'y'], s=6, alpha=0.6,
                       color=('#c7ccd3' if c == 'Unassigned' else color), label=c, linewidths=0)
        ax.legend(fontsize=6, markerscale=1.4, loc='center left', bbox_to_anchor=(1.0, 0.5))
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, axes = plt.subplots(2, 2, figsize=(17, 13))
_scat(axes[0, 0], 'parent', 'by parent theme')
_scat(axes[0, 1], 'college_grp', 'by lead-PI college')
_scat(axes[1, 0], 'agency_grp', 'by funding agency')
_scat(axes[1, 1], 'startdateyear', 'by start year', cont=True)
fig.suptitle('SPECTER2 + UMAP projection of NEU grants', fontsize=14)
plt.tight_layout(); plt.savefig(FIG_DIR / 'w7_umap_grants_specter2.png', dpi=120, bbox_inches='tight'); plt.show()

### Interactive projection (coloured by topic) → `docs/07_grant_projection_specter2.html`

In [ ]:
pdf = udf.copy()
pdf['topic_label'] = pdf.topic_label.fillna('Unassigned')
pdf['title'] = pdf.grantname.astype(str).str[:110]
fig = px.scatter(pdf, x='x', y='y', color='topic_label', hover_name='title',
                 hover_data={'x': False, 'y': False, 'agencyname': True,
                             'startdateyear': True, 'college': True},
                 title='SPECTER2 + UMAP · NEU grants coloured by BERTopic topic',
                 width=1050, height=720, opacity=0.65)
fig.update_traces(marker=dict(size=5))
fig.update_layout(legend=dict(font=dict(size=9)))
fig.write_html(DOCS / '07_grant_projection_specter2.html')
print('wrote', DOCS / '07_grant_projection_specter2.html')
fig.show()

## 7 · Topics over time, by agency, and by funding

In [ ]:
tg = g[g.topic_id >= 0].copy()
# funding per topic ($M) + avg grant size
fund = tg.groupby('topic_label')['totaldollars'].agg(['sum', 'mean', 'size'])
fund['sum'] /= 1e6; fund['mean'] /= 1e6
fund = fund.sort_values('sum', ascending=False)
fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(range(len(fund)), fund['sum'], color='darkgreen')
ax.set_yticks(range(len(fund))); ax.set_yticklabels(fund.index, fontsize=8); ax.invert_yaxis()
ax.set_xlabel('total funding ($M)'); ax.set_title('Funding by topic')
plt.tight_layout(); plt.savefig(FIG_DIR / 'w7_topic_funding.png', dpi=120, bbox_inches='tight'); plt.show()

# topics over time (share of year), 2005+
tt = tg[tg.startdateyear >= 2005]
trend = pd.crosstab(tt.startdateyear, tt.parent)
trend_pct = trend.div(trend.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=(11, 6))
for c in trend_pct.columns:
    ax.plot(trend_pct.index, trend_pct[c], marker='o', markersize=3, linewidth=1.5, label=c)
ax.set_xlabel('year'); ax.set_ylabel("share of year's grants (%)")
ax.set_title('Parent-theme mix over time (2005+)')
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig(FIG_DIR / 'w7_parent_over_time.png', dpi=120, bbox_inches='tight'); plt.show()

## Deliverables

- Figures (`notebooks/figures/`): `w7_topic_prevalence`, `w7_topic_by_college_rownorm`,
  `w7_parent_sizes`, `w7_topic_dendrogram`, `w7_umap_grants_specter2`, `w7_topic_funding`,
  `w7_parent_over_time`.
- `outputs/college_profiles.csv`
- `docs/07_grant_projection_specter2.html` — interactive projection coloured by BERTopic topic.

All driven by BERTopic's fit (historical/comparison as of 2026-08-29 — see the note at the top of this notebook); the LDA path is retired (see
`src/topics_lda.py` for the legacy fit and `TOPIC_WORK_FORWARD_PLAN.md` M5b for the
planned LDA-vs-BERTopic agreement crosstab).